# wandb-log-step — worked example 2: Log loss, accuracy, and lr on the same step

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `wandb-log-step`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

You can log multiple metrics in a single `wandb.log` call by putting them all in the same dict. This is cleaner than multiple `wandb.log` calls for the same step and avoids the subtle `commit` semantics you'd need otherwise. All metrics in the dict are recorded at the same `step` value.

## Worked solution

**Step 1 — accumulate examples_seen.**
As before, we maintain `examples_seen` and increment by `batch_size` each iteration before logging.

**Step 2 — build a multi-metric dict.**
Instead of logging loss alone, we build a dict with all the metrics we want to record for this step: `{'loss': ..., 'accuracy': ..., 'lr': ...}`. All three will be recorded at the same x-axis position.

**Step 3 — single wandb.log call per step.**
We call `wandb.log(metrics_dict, step=examples_seen)` once per batch. This produces one `wandb.log` call per step on the dashboard, with all metrics grouped together, rather than three separate calls that wandb might assign to different step values.

In [ ]:
import sys
from unittest.mock import MagicMock
sys.modules.setdefault('wandb', MagicMock())
import wandb

def log_multi_metric_loop(step_data, batch_size):
    """
    step_data: list of dicts, each with keys 'loss', 'accuracy', 'lr'
    batch_size: examples per step
    Returns: final examples_seen
    """
    examples_seen = 0
    for data in step_data:
        examples_seen += batch_size
        wandb.log(
            {
                'train/loss': data['loss'],
                'train/accuracy': data['accuracy'],
                'train/lr': data['lr'],
            },
            step=examples_seen
        )
    return examples_seen

# Exercise it
wandb.log.reset_mock()
step_data = [
    {'loss': 2.3, 'accuracy': 0.12, 'lr': 1e-3},
    {'loss': 1.8, 'accuracy': 0.31, 'lr': 9e-4},
    {'loss': 1.2, 'accuracy': 0.55, 'lr': 8e-4},
]
final = log_multi_metric_loop(step_data, batch_size=64)
print('Final examples_seen:', final)  # 192
for i, call in enumerate(wandb.log.call_args_list):
    print(f'  step {call.kwargs["step"]}: {list(call.args[0].keys())}')